# 01 - Statistical Analysis with SciPy

This notebook starts from the basics and then moves into Module 5 statistical analysis requirements.

You will learn how to:

1. Load financial indicator data from SQL Server when available.
2. Fall back to generated training data when running in Google Colab or without SQL Server.
3. Calculate descriptive statistics.
4. Run correlation analysis.
5. Run a hypothesis test and interpret the result.

In [ ]:
# Install required libraries for a fresh notebook environment such as Google Colab.
import sys
import subprocess

packages = ["pandas", "numpy", "scipy", "sqlalchemy", "pyodbc", "python-dotenv", "openpyxl"]

for package in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    except Exception as exc:
        print(f"Package install skipped or failed for {package}: {exc}")

In [ ]:
# Imports and shared notebook setup.
import os
from pathlib import Path
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
from scipy import stats

try:
    import pyodbc
except Exception as exc:
    pyodbc = None
    print("pyodbc unavailable; SQL Server extraction will be skipped:", exc)

try:
    from dotenv import load_dotenv
    from sqlalchemy import create_engine
except Exception as exc:
    load_dotenv = None
    create_engine = None
    print("SQLAlchemy/dotenv unavailable; SQL Server extraction will be skipped:", exc)

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
ENV_FILE = ".env"  # Use ".env.windows" on Windows when running locally.

print("Working directory:", BASE_DIR)

In [ ]:
# Beginner note:
# A DataFrame is a table-like object in pandas. Each column has a name and each row is one observation.
# This fallback generator creates the same kind of financial supervision data as the SQL setup script.
def build_sample_indicator_data():
    institutions = [
        ("CBL", "Central Bank Liquidity Desk", "Maseru", "Central Bank", 920_000_000, 310_000_000, 0.345, 0.218, 0.020, 1),
        ("MCB", "Maseru Commercial Bank", "Maseru", "Commercial Bank", 610_000_000, 520_000_000, 0.278, 0.161, 0.048, 2),
        ("LMB", "Leribe Microfinance Bank", "Leribe", "Microfinance", 185_000_000, 171_000_000, 0.235, 0.139, 0.073, 3),
        ("QFB", "Quthing Finance Bank", "Quthing", "Commercial Bank", 275_000_000, 251_000_000, 0.242, 0.148, 0.067, 4),
        ("BDB", "Butha-Buthe Development Bank", "Butha-Buthe", "Development Bank", 330_000_000, 298_000_000, 0.255, 0.154, 0.061, 5),
        ("MFI", "Mafeteng Inclusion Finance", "Mafeteng", "Microfinance", 145_000_000, 139_000_000, 0.218, 0.128, 0.086, 6),
    ]
    rows = []
    observation_id = 1
    for day_number, observation_date in enumerate(pd.date_range("2026-04-01", periods=60, freq="D")):
        inflation = round(0.0470 + ((day_number % 11) * 0.0007), 4)
        interbank = round(0.0820 + ((day_number % 7) * 0.0009), 4)
        for code, name, region, inst_type, deposits, loans, liquidity, capital, npl, weight in institutions:
            liquidity_ratio = round(liquidity + ((day_number % 10) * 0.0021) - (0.014 if weight in (3, 6) else 0) - (0.018 if 33 <= day_number <= 42 else 0), 4)
            capital_ratio = round(capital + ((day_number % 8) * 0.0014) - (0.011 if weight == 6 else 0), 4)
            npl_ratio = round(npl + ((day_number % 9) * 0.0018) + (0.010 if 33 <= day_number <= 42 else 0), 4)
            credit_growth = round(0.0310 + ((day_number % 12) * 0.0016) - (npl * 0.0800), 4)
            stress_flag = int(liquidity_ratio < 0.2300 or capital_ratio < 0.1300 or npl_ratio >= 0.0850 or credit_growth < 0.0280)
            rows.append({
                "ObservationID": observation_id,
                "ObservationDate": observation_date,
                "InstitutionCode": code,
                "InstitutionName": name,
                "Region": region,
                "InstitutionType": inst_type,
                "TotalDepositsLSL": deposits + (day_number * 625_000) + (weight * 34_000) + ((day_number % 6) * 820_000),
                "TotalLoansLSL": loans + (day_number * 475_000) + (weight * 27_000) + ((day_number % 5) * 550_000),
                "LiquidityRatio": liquidity_ratio,
                "CapitalAdequacyRatio": capital_ratio,
                "NplRatio": npl_ratio,
                "TransactionVolume": 850 + (weight * 95) + (day_number * 8) + ((day_number % 6) * 35),
                "TransactionValueLSL": (deposits * 0.028) + (day_number * 185_000) + (weight * 75_000) + ((day_number % 7) * 430_000),
                "CreditGrowthRate": credit_growth,
                "InflationRate": inflation,
                "InterbankRate": interbank,
                "StressFlag": stress_flag,
            })
            observation_id += 1
    return pd.DataFrame(rows)

def get_sqlalchemy_engine(env_file=ENV_FILE):
    if pyodbc is None or load_dotenv is None or create_engine is None:
        raise RuntimeError("SQL Server packages are unavailable")
    load_dotenv(BASE_DIR / env_file)
    driver = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
    server = os.getenv("DB_SERVER", "localhost,1433")
    database = os.getenv("DB_NAME", "TrainingDB")
    user = os.getenv("DB_USER", "sa")
    password = os.getenv("DB_PASSWORD", "StrongPassw0rd!2026")
    trusted = os.getenv("DB_TRUSTED", "no").lower() in ("yes", "true", "1")
    parts = [f"DRIVER={{{driver}}}", f"SERVER={server}", f"DATABASE={database}", "Encrypt=yes", "TrustServerCertificate=yes", "Connection Timeout=5"]
    if trusted:
        parts.append("Trusted_Connection=yes")
    else:
        parts.extend([f"UID={user}", f"PWD={password}"])
    return create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(';'.join(parts) + ';')}")

def load_indicator_data():
    query = """
    SELECT
        ObservationID, ObservationDate, InstitutionCode, InstitutionName, Region, InstitutionType,
        TotalDepositsLSL, TotalLoansLSL, LiquidityRatio, CapitalAdequacyRatio, NplRatio,
        TransactionVolume, TransactionValueLSL, CreditGrowthRate, InflationRate, InterbankRate,
        CAST(StressFlag AS INT) AS StressFlag
    FROM m5.DailyFinancialIndicators
    ORDER BY ObservationDate, InstitutionCode;
    """
    try:
        engine = get_sqlalchemy_engine()
        data = pd.read_sql(query, engine, parse_dates=["ObservationDate"])
        source = "SQL Server"
    except Exception as exc:
        print("Using generated sample data. SQL Server unavailable:", exc)
        data = build_sample_indicator_data()
        source = "generated fallback data"
    numeric_columns = ["TotalDepositsLSL", "TotalLoansLSL", "LiquidityRatio", "CapitalAdequacyRatio", "NplRatio", "TransactionValueLSL", "CreditGrowthRate", "InflationRate", "InterbankRate"]
    data[numeric_columns] = data[numeric_columns].astype(float)
    print("Data source:", source)
    return data

df = load_indicator_data()
display(df.head())
print("Rows:", len(df), "Columns:", len(df.columns))

## 1. Descriptive Statistics

Descriptive statistics summarize a dataset. Common measures are:

- `mean`: average value
- `median`: middle value
- `std`: standard deviation, or spread around the mean
- `min` / `max`: smallest and largest values

In [ ]:
analysis_columns = [
    "LiquidityRatio", "CapitalAdequacyRatio", "NplRatio", "TransactionValueLSL",
    "CreditGrowthRate", "InflationRate", "InterbankRate",
]

summary = df[analysis_columns].describe().T
summary["variance"] = df[analysis_columns].var(numeric_only=True)
summary["median"] = df[analysis_columns].median(numeric_only=True)
summary.to_csv(OUTPUT_DIR / "statistical_summary.csv")
display(summary)

print("Interpretation example:")
print("A higher NPL ratio means a larger share of loans are non-performing, which can indicate credit risk.")

## 2. Correlation Analysis

Correlation measures how two numeric variables move together.

- Close to `+1`: they rise together.
- Close to `-1`: one rises while the other falls.
- Close to `0`: little linear relationship.

Correlation does not prove causation.

In [ ]:
correlations = df[analysis_columns].corr()
liquidity_npl_corr, liquidity_npl_pvalue = stats.pearsonr(df["LiquidityRatio"], df["NplRatio"])

print(f"Liquidity vs NPL correlation: {liquidity_npl_corr:.3f}")
print(f"p-value: {liquidity_npl_pvalue:.6f}")

if abs(liquidity_npl_corr) >= 0.7:
    print("Interpretation: strong linear relationship.")
elif abs(liquidity_npl_corr) >= 0.3:
    print("Interpretation: moderate linear relationship.")
else:
    print("Interpretation: weak linear relationship.")

correlations.to_csv(OUTPUT_DIR / "correlation_matrix.csv")
display(correlations)

## 3. Hypothesis Testing

A hypothesis test asks whether an observed difference is likely to be real or just random variation.

Question: is average credit growth different between stressed and non-stressed observations?

- Null hypothesis: the two groups have the same average credit growth.
- Alternative hypothesis: the two groups have different average credit growth.
- If `p-value < 0.05`, we reject the null hypothesis for this training exercise.

In [ ]:
stressed = df.loc[df["StressFlag"] == 1, "CreditGrowthRate"]
not_stressed = df.loc[df["StressFlag"] == 0, "CreditGrowthRate"]

# Welch's t-test does not assume the two groups have equal variance.
test_result = stats.ttest_ind(stressed, not_stressed, equal_var=False)

print(f"Stressed mean credit growth: {stressed.mean():.4f}")
print(f"Non-stressed mean credit growth: {not_stressed.mean():.4f}")
print(f"t-statistic: {test_result.statistic:.3f}")
print(f"p-value: {test_result.pvalue:.6f}")

if test_result.pvalue < 0.05:
    print("Decision: reject the null hypothesis. The group averages are statistically different in this dataset.")
else:
    print("Decision: do not reject the null hypothesis. The observed difference is not statistically significant here.")

## 4. Group Comparison for Reporting

Group summaries help turn statistical analysis into business insight.

In [ ]:
group_summary = (
    df.groupby("InstitutionType")
    .agg(
        rows=("ObservationID", "count"),
        mean_liquidity=("LiquidityRatio", "mean"),
        mean_npl=("NplRatio", "mean"),
        stress_rate=("StressFlag", "mean"),
        avg_transaction_value=("TransactionValueLSL", "mean"),
    )
    .sort_values("stress_rate", ascending=False)
)

group_summary.to_csv(OUTPUT_DIR / "institution_type_summary.csv")
display(group_summary)

print("Saved outputs:")
print(OUTPUT_DIR / "statistical_summary.csv")
print(OUTPUT_DIR / "correlation_matrix.csv")
print(OUTPUT_DIR / "institution_type_summary.csv")